# Streamlit Deployment from Jupyter Notebook

This notebook helps you deploy your completed Sentinel-2 classification results using **Streamlit**.

## 1. Required packages

In [1]:
!pip install streamlit pandas numpy plotly pillow

In [2]:
from pathlib import Path
import zipfile
import shutil
import os
import sys
import subprocess
import time
import webbrowser
import pandas as pd
import socket



# # # Folder where this notebook is running
NOTEBOOK_FOLDER = Path.cwd()


APP_DIR = NOTEBOOK_FOLDER 


In [3]:
import json

CREATE_APP_FROM_NOTEBOOK = False  

APP_PY_CODE = ''
REQUIREMENTS_TEXT = 'streamlit\npandas\nnumpy\nplotly\npillow\n'

if CREATE_APP_FROM_NOTEBOOK:
    APP_DIR.mkdir(parents=True, exist_ok=True)
    (APP_DIR / "data").mkdir(parents=True, exist_ok=True)
    (APP_DIR / "images" / "confusion_matrices").mkdir(parents=True, exist_ok=True)
    (APP_DIR / "images" / "classified_maps").mkdir(parents=True, exist_ok=True)
    (APP_DIR / "metadata").mkdir(parents=True, exist_ok=True)

    (APP_DIR / "app.py").write_text(APP_PY_CODE, encoding="utf-8")
    (APP_DIR / "requirements.txt").write_text(REQUIREMENTS_TEXT, encoding="utf-8")

    readme_text = '''
# Sentinel-2 LULC Streamlit Dashboard

Run locally with:

streamlit run app.py

Main files required:
- data/all_model_metrics.csv
- data/all_area_statistics.csv
- data/all_feature_importance_statistics.csv
- data/class_code_mapping.csv
- images/confusion_matrices/
- images/classified_maps/
'''
    (APP_DIR / "README.md").write_text(readme_text.strip(), encoding="utf-8")

    metadata = {
        "project": "Sentinel-2 LULC classification Streamlit dashboard",
        "note": "Created from Jupyter notebook. Copy your CSV and PNG results into the data and images folders."
    }
    (APP_DIR / "metadata" / "metadata.json").write_text(json.dumps(metadata, indent=4), encoding="utf-8")

    print("App files created at:", APP_DIR)
else:
    print("Skipped. Set CREATE_APP_FROM_NOTEBOOK = True only if you do not have the zip package.")

Skipped. Set CREATE_APP_FROM_NOTEBOOK = True only if you do not have the zip package.


## 2. Check that your app files are present

This checks the files needed by the dashboard.

The app can still run if one or two images are missing, but the main CSV files should be present.

In [4]:
required_files = [
    APP_DIR / "app.py",
    APP_DIR / "requirements.txt",
    APP_DIR / "data" / "all_model_metrics.csv",
    APP_DIR / "data" / "all_area_statistics.csv",
    APP_DIR / "data" / "all_feature_importance_statistics.csv",
    APP_DIR / "data" / "class_code_mapping.csv",
]

check_rows = []
for file in required_files:
    check_rows.append({
        "file": str(file.relative_to(APP_DIR)) if APP_DIR in file.parents else str(file),
        "exists": file.exists()
    })

check_df = pd.DataFrame(check_rows)
display(check_df)

confusion_dir = APP_DIR / "images" / "confusion_matrices"
maps_dir = APP_DIR / "images" / "classified_maps"

confusion_count = len(list(confusion_dir.glob("*.png"))) if confusion_dir.exists() else 0
map_count = len(list(maps_dir.glob("*.png"))) if maps_dir.exists() else 0

print("Confusion matrix PNG count:", confusion_count)
print("Classified map PNG count:", map_count)

if not (APP_DIR / "app.py").exists():
    raise FileNotFoundError("app.py was not found. Extract the zip or create the app files first.")

,file,exists
0,app.py,True
1,requirements.txt,True
2,data\all_model_metrics.csv,True
3,data\all_area_statistics.csv,True
4,data\all_feature_importance_statistics.csv,True
5,data\class_code_mapping.csv,True


Confusion matrix PNG count: 12
Classified map PNG count: 12


## 3. Preview the CSV tables inside Jupyter

This confirms that Streamlit will be able to read your results.

In [5]:
data_files = {
    "Metrics": APP_DIR / "data" / "all_model_metrics.csv",
    "Area statistics": APP_DIR / "data" / "all_area_statistics.csv",
    "Feature importance": APP_DIR / "data" / "all_feature_importance_statistics.csv",
    "Class mapping": APP_DIR / "data" / "class_code_mapping.csv",
}

for name, path in data_files.items():
    print("\n" + "=" * 70)
    print(name)
    print(path)
    if path.exists():
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        display(df.head())
    else:
        print("Missing")


Metrics
C:\Users\N1386471\OneDrive - Nottingham Trent University\Documents\GitHub\LULC_Classification_NTU\data\all_model_metrics.csv
Shape: (12, 10)


,Year,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1-Score,MCC,Mean IoU,Source
0,2017,FT-Transformer,0.99,0.98,0.99,0.98,0.98,0.98,0.97,derived from uploaded confusion matrix images
1,2017,Random Forest,0.98,0.97,0.99,0.97,0.98,0.98,0.96,derived from uploaded confusion matrix images
2,2017,SVM,0.98,0.97,0.98,0.97,0.98,0.97,0.95,derived from uploaded confusion matrix images
3,2017,TabNet,0.99,0.99,0.99,0.99,0.99,0.99,0.98,derived from uploaded confusion matrix images
4,2021,FT-Transformer,0.97,0.97,0.96,0.97,0.97,0.96,0.94,derived from uploaded confusion matrix images



Area statistics
C:\Users\N1386471\OneDrive - Nottingham Trent University\Documents\GitHub\LULC_Classification_NTU\data\all_area_statistics.csv
Shape: (60, 8)


,Year,Model,map_code,original_class_label,pixel_count,area_m2,area_ha,area_km2
0,2017,RF,1,BL,10257,1025700,102.6,1.0
1,2017,RF,2,BU,477855,47785500,4778.6,47.8
2,2017,RF,3,GL,93549,9354900,935.5,9.4
3,2017,RF,4,TR,149584,14958400,1495.8,15.0
4,2017,RF,5,WB,15222,1522200,152.2,1.5



Feature importance
C:\Users\N1386471\OneDrive - Nottingham Trent University\Documents\GitHub\LULC_Classification_NTU\data\all_feature_importance_statistics.csv
Shape: (132, 4)


,Year,Model_Display,feature,importance
0,2017,RF,NDVI,0.14
1,2017,RF,B04,0.13
2,2017,RF,B11,0.13
3,2017,RF,B08,0.11
4,2017,RF,B03,0.10



Class mapping
C:\Users\N1386471\OneDrive - Nottingham Trent University\Documents\GitHub\LULC_Classification_NTU\data\class_code_mapping.csv
Shape: (5, 3)


,map_code,encoded_label,original_class_label
0,1,0,Bare land
1,2,1,Built up
2,3,2,Grassland
3,4,3,Tree
4,5,4,Waterbody


## 4. Deploy

In [7]:
APP_DIR = Path(
    r"C:\Users\N1386471\OneDrive - Nottingham Trent University\Documents\GitHub\LULC_Classification_NTU"
)

APP_FILE = APP_DIR / "app.py"

if not APP_FILE.exists():
    raise FileNotFoundError(
        f"app.py was not found here:\n{APP_FILE}\n\n"
        "Check that APP_DIR points to the folder that contains app.py."
    )

print("App folder found:")
print(APP_DIR)

App folder found:
C:\Users\N1386471\OneDrive - Nottingham Trent University\Documents\GitHub\LULC_Classification_NTU


In [8]:
streamlit_config_dir = APP_DIR / ".streamlit"
streamlit_config_dir.mkdir(parents=True, exist_ok=True)

config_file = streamlit_config_dir / "config.toml"

config_file.write_text(
    """
[server]
headless = true
showEmailPrompt = false
port = 8501

[browser]
gatherUsageStats = false
""".strip(),
    encoding="utf-8"
)

print("Streamlit config written to:")
print(config_file)

Streamlit config written to:
C:\Users\N1386471\OneDrive - Nottingham Trent University\Documents\GitHub\LULC_Classification_NTU\.streamlit\config.toml


In [9]:
home_streamlit_dir = Path.home() / ".streamlit"
home_streamlit_dir.mkdir(parents=True, exist_ok=True)

credentials_file = home_streamlit_dir / "credentials.toml"

credentials_file.write_text(
    """
[general]
email = ""
""".strip(),
    encoding="utf-8"
)

print("Streamlit credentials written to:")
print(credentials_file)


Streamlit credentials written to:
C:\Users\N1386471\.streamlit\credentials.toml


In [10]:
def port_is_busy(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

PORT = 8501

if port_is_busy(PORT):
    print(f"Port {PORT} is already busy. Using port 8502 instead.")
    PORT = 8502



In [11]:
log_path = APP_DIR / "streamlit_log.txt"

# Close old process if it exists
try:
    streamlit_process.terminate()
    time.sleep(2)
except Exception:
    pass

env = os.environ.copy()
env["STREAMLIT_SERVER_HEADLESS"] = "true"
env["STREAMLIT_BROWSER_GATHER_USAGE_STATS"] = "false"
env["STREAMLIT_SERVER_SHOW_EMAIL_PROMPT"] = "false"

cmd = [
    sys.executable,
    "-m",
    "streamlit",
    "run",
    "app.py",
    "--server.port",
    str(PORT),
    "--server.headless",
    "true",
    "--server.showEmailPrompt",
    "false",
    "--browser.gatherUsageStats",
    "false"
]

print("\nStarting Streamlit with this command:")
print(" ".join(cmd))

log_file = open(log_path, "w", encoding="utf-8")

streamlit_process = subprocess.Popen(
    cmd,
    cwd=str(APP_DIR),
    stdout=log_file,
    stderr=subprocess.STDOUT,
    stdin=subprocess.DEVNULL,
    text=True,
    env=env
)

time.sleep(8)
log_file.flush()


Starting Streamlit with this command:
C:\ProgramData\anaconda3\python.exe -m streamlit run app.py --server.port 8501 --server.headless true --server.showEmailPrompt false --browser.gatherUsageStats false


In [12]:
if streamlit_process.poll() is None:
    url = f"http://127.0.0.1:{PORT}"
    print("\nStreamlit is running successfully.")
    print("Open this link:")
    print(url)

    try:
        webbrowser.open(url)
    except Exception:
        pass

else:
    print("\nStreamlit stopped because there is still an error.")
    print("Open this log file:")
    print(log_path)

    try:
        print("\nLast log lines:")
        print(log_path.read_text(encoding="utf-8")[-4000:])
    except Exception as e:
        print("Could not read log file:", e)


Streamlit is running successfully.
Open this link:
http://127.0.0.1:8501
